# SignLearn-09 — Final Comparative Holdout Audit

SignLearn-08B passed its validation promotion gate. This notebook performs **no training**
and makes no model-selection changes. It compares the frozen production 189-feature model
with the frozen 492-feature candidate on the same Google holdout signers.

The decision rules below are declared before inference:

- candidate holdout macro F1 must be at least as high as production;
- candidate holdout accuracy may be at most 0.10 percentage points lower;
- candidate mirrored-view macro F1 may be at most 0.50 points lower;
- all tensors, signer splits, labels, hashes, and feature contracts must pass validation.

The Google holdout has appeared in earlier project reports, so it is described honestly as
an **internal comparative holdout**, not a pristine external test. WLASL remains untouched.

Attach three Kaggle inputs: the temporal NPZ cache, the current 189-feature mirror-robust
bundle, and `signlearn_08b_candidate_bundle.zip`.

In [ ]:
from pathlib import Path, PurePosixPath
from IPython.display import display
import hashlib
import json
import shutil
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, top_k_accuracy_score

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
SEED = 42
BATCH_SIZE = 128
MINIMUM_TEST_MACRO_F1_GAIN = 0.0
TEST_ACCURACY_TOLERANCE = 0.001
TEST_MIRROR_F1_TOLERANCE = 0.005

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

KAGGLE_INPUT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/signlearn_09')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Frozen decision rules:', {
    'minimum_test_macro_f1_gain': MINIMUM_TEST_MACRO_F1_GAIN,
    'test_accuracy_tolerance': TEST_ACCURACY_TOLERANCE,
    'test_mirror_f1_tolerance': TEST_MIRROR_F1_TOLERANCE,
})

## 1. Resolve the production and candidate bundles by manifest contract

In [ ]:
def manifest_feature_count(manifest):
    shape = manifest.get('input_shape', [])
    return int(shape[-1]) if shape else None

bundle_candidates = []
for manifest_path in KAGGLE_INPUT.rglob('deployment_manifest.json'):
    try:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    except Exception:
        continue
    model_path = manifest_path.parent / 'signlearn_model.keras'
    if model_path.exists() and manifest_feature_count(manifest) in {189, 492}:
        evidence = (str(manifest_path) + json.dumps(manifest)).lower()
        score = 50 * (manifest_path.parent / 'camera_orientation_policy.json').exists()
        score += 30 * ('mirror' in evidence) + 30 * ('transfer' in evidence)
        bundle_candidates.append({
            'count': manifest_feature_count(manifest), 'score': score,
            'kind': 'directory', 'location': manifest_path.parent, 'manifest': manifest,
        })

for zip_path in KAGGLE_INPUT.rglob('*.zip'):
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = set(archive.namelist())
            for member in names:
                if not member.endswith('deployment_manifest.json'):
                    continue
                try:
                    manifest = json.loads(archive.read(member).decode('utf-8'))
                except Exception:
                    continue
                count = manifest_feature_count(manifest)
                parent = PurePosixPath(member).parent
                model_member = str(parent / 'signlearn_model.keras')
                if count not in {189, 492} or model_member not in names:
                    continue
                camera_member = str(parent / 'camera_orientation_policy.json')
                evidence = (str(zip_path) + member + json.dumps(manifest)).lower()
                score = 50 * (camera_member in names)
                score += 30 * ('mirror' in evidence) + 30 * ('transfer' in evidence)
                bundle_candidates.append({
                    'count': count, 'score': score, 'kind': 'zip',
                    'location': (zip_path, str(parent)), 'manifest': manifest,
                })
    except zipfile.BadZipFile:
        continue

def select_and_materialize(feature_count, label):
    eligible = [item for item in bundle_candidates if item['count'] == feature_count]
    if not eligible:
        raise FileNotFoundError(f'No {feature_count}-feature model bundle found in Kaggle inputs.')
    selected = sorted(eligible, key=lambda item: item['score'], reverse=True)[0]
    if selected['kind'] == 'directory':
        root = selected['location']
    else:
        zip_path, member_parent = selected['location']
        destination = OUTPUT_DIR / f'input_{label}'
        if destination.exists():
            shutil.rmtree(destination)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(destination)
        root = destination / Path(member_parent)
    return root, selected['manifest'], selected['score']

SOURCE_ROOT, source_manifest, source_score = select_and_materialize(189, 'production')
CANDIDATE_ROOT, candidate_manifest, candidate_score = select_and_materialize(492, 'candidate')
SOURCE_MODEL_PATH = SOURCE_ROOT / 'signlearn_model.keras'
CANDIDATE_MODEL_PATH = CANDIDATE_ROOT / 'signlearn_model.keras'
LABEL_MAP_PATH = SOURCE_ROOT / 'label_map.json'
BASE_FEATURE_NAMES_PATH = SOURCE_ROOT / 'frame_feature_names.json'
CANDIDATE_FEATURE_NAMES_PATH = CANDIDATE_ROOT / 'frame_feature_names.json'

cache_candidates = [
    path for path in KAGGLE_INPUT.rglob('mvp50_temporal_48_frames.npz')
    if 'wlasl' not in str(path).lower()
]
if not cache_candidates:
    raise FileNotFoundError('Attach mvp50_temporal_48_frames.npz.')
CACHE_PATH = cache_candidates[0]
print('Production bundle:', SOURCE_ROOT, 'score:', source_score)
print('Candidate bundle:', CANDIDATE_ROOT, 'score:', candidate_score)
print('Temporal cache:', CACHE_PATH)

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

cache = np.load(CACHE_PATH, allow_pickle=False)
X = cache['X'].astype(np.float32)
y = cache['y'].astype(np.int32)
splits = cache['splits'].astype(str)
participant_ids = cache['participant_ids']
sequence_ids = cache['sequence_ids']
with LABEL_MAP_PATH.open('r', encoding='utf-8') as stream:
    label_map = json.load(stream)
with BASE_FEATURE_NAMES_PATH.open('r', encoding='utf-8') as stream:
    base_feature_names = json.load(stream)
with CANDIDATE_FEATURE_NAMES_PATH.open('r', encoding='utf-8') as stream:
    candidate_feature_names = json.load(stream)

train_mask = splits == 'train'
validation_mask = splits == 'validation'
test_mask = splits == 'test'
X_validation = X[validation_mask]  # used only for a feature-contract sample
X_test, y_test = X[test_mask], y[test_mask]
participant_sets = {
    name: set(participant_ids[splits == name].tolist())
    for name in ['train', 'validation', 'test']
}
assert participant_sets['train'].isdisjoint(participant_sets['validation'])
assert participant_sets['train'].isdisjoint(participant_sets['test'])
assert participant_sets['validation'].isdisjoint(participant_sets['test'])
assert X.shape[1:] == (48, 189)
assert len(base_feature_names) == 189
assert len(candidate_feature_names) == 492
assert sorted(label_map.values()) == list(range(50))
assert candidate_manifest.get('google_test_status') == 'not evaluated'
assert 'wlasl' not in str(CACHE_PATH).lower()
print('Production SHA-256:', sha256(SOURCE_MODEL_PATH))
print('Candidate SHA-256:', sha256(CANDIDATE_MODEL_PATH))
print('Candidate expected SHA-256:', candidate_manifest.get('candidate_model_sha256', 'manifest omitted hash'))
print('Holdout sequences:', len(X_test), 'holdout signers:', len(participant_sets['test']))
print('WLASL loaded: False')

## 2. Rebuild the frozen 492-feature contract

In [ ]:
BASE_FEATURE_COUNT = 189
MOTION_COORDINATE_COUNT = 144
GEOMETRY_FEATURE_NAMES = [
    'wrist_to_wrist_distance',
    'left_wrist_to_mouth_distance', 'right_wrist_to_mouth_distance',
    'left_wrist_to_nose_distance', 'right_wrist_to_nose_distance',
    'left_wrist_to_shoulder_distance', 'right_wrist_to_shoulder_distance',
    'left_hand_spread', 'right_hand_spread',
    'left_thumb_index_distance', 'right_thumb_index_distance',
    'left_hand_to_pose_wrist_distance', 'right_hand_to_pose_wrist_distance',
    'left_fingertip_to_mouth_min_distance', 'right_fingertip_to_mouth_min_distance',
]
ENHANCED_FEATURE_COUNT = BASE_FEATURE_COUNT + 2 * MOTION_COORDINATE_COUNT + len(GEOMETRY_FEATURE_NAMES)
EXTRA_FEATURE_COUNT = ENHANCED_FEATURE_COUNT - BASE_FEATURE_COUNT
assert ENHANCED_FEATURE_COUNT == 492

reflection_sign = np.ones(BASE_FEATURE_COUNT, dtype=np.float32)
reflection_sign[0:63:3] = -1.0
reflection_sign[63:126:3] = -1.0
reflection_sign[126:144:2] = -1.0
reflection_sign[144:184:2] = -1.0
mirror_permutation = np.arange(BASE_FEATURE_COUNT, dtype=np.int32)
mirror_permutation[0:63] = np.arange(63, 126)
mirror_permutation[63:126] = np.arange(0, 63)
mirror_permutation[184], mirror_permutation[185] = 185, 184
TF_REFLECTION_SIGN = tf.constant(reflection_sign, tf.float32)
TF_MIRROR_PERMUTATION = tf.constant(mirror_permutation, tf.int32)

def mirror_base_features(features):
    return tf.gather(features, TF_MIRROR_PERMUTATION, axis=-1) * TF_REFLECTION_SIGN

def vector_distance(first, second):
    return tf.sqrt(tf.reduce_sum(tf.square(first - second), axis=-1) + 1e-8)

def build_enhanced_features(features):
    features = tf.cast(features, tf.float32)
    motion_coordinates = features[:, :MOTION_COORDINATE_COUNT]
    detection = features[:, 184:189]

    previous_coordinates = tf.concat([motion_coordinates[:1], motion_coordinates[:-1]], axis=0)
    velocity = motion_coordinates - previous_coordinates
    previous_detection = tf.concat([detection[:1], detection[:-1]], axis=0)
    velocity_masks = tf.concat([
        tf.repeat((detection[:, 0:1] > 0.5) & (previous_detection[:, 0:1] > 0.5), 63, axis=1),
        tf.repeat((detection[:, 1:2] > 0.5) & (previous_detection[:, 1:2] > 0.5), 63, axis=1),
        tf.repeat((detection[:, 2:3] > 0.5) & (previous_detection[:, 2:3] > 0.5), 18, axis=1),
    ], axis=1)
    velocity = tf.where(velocity_masks, velocity, 0.0)
    previous_velocity = tf.concat([velocity[:1], velocity[:-1]], axis=0)
    acceleration = velocity - previous_velocity
    previous_velocity_mask = tf.concat([velocity_masks[:1], velocity_masks[:-1]], axis=0)
    acceleration = tf.where(velocity_masks & previous_velocity_mask, acceleration, 0.0)
    velocity = tf.clip_by_value(velocity, -5.0, 5.0)
    acceleration = tf.clip_by_value(acceleration, -5.0, 5.0)

    left_hand = tf.reshape(features[:, 0:63], [-1, 21, 3])
    right_hand = tf.reshape(features[:, 63:126], [-1, 21, 3])
    pose = tf.reshape(features[:, 126:144], [-1, 9, 2])
    lips = tf.reshape(features[:, 144:184], [-1, 20, 2])
    left_wrist, right_wrist = left_hand[:, 0, :2], right_hand[:, 0, :2]
    nose, left_shoulder, right_shoulder = pose[:, 0], pose[:, 1], pose[:, 2]
    left_pose_wrist, right_pose_wrist = pose[:, 5], pose[:, 6]
    mouth = tf.reduce_mean(lips, axis=1)
    fingertip_indices = tf.constant([4, 8, 12, 16, 20], tf.int32)
    left_tips = tf.gather(left_hand[:, :, :2], fingertip_indices, axis=1)
    right_tips = tf.gather(right_hand[:, :, :2], fingertip_indices, axis=1)
    left_spread = tf.reduce_mean(vector_distance(left_tips, left_wrist[:, None, :]), axis=1)
    right_spread = tf.reduce_mean(vector_distance(right_tips, right_wrist[:, None, :]), axis=1)
    left_tip_to_mouth = tf.reduce_min(vector_distance(left_tips, mouth[:, None, :]), axis=1)
    right_tip_to_mouth = tf.reduce_min(vector_distance(right_tips, mouth[:, None, :]), axis=1)

    geometry = tf.stack([
        vector_distance(left_wrist, right_wrist),
        vector_distance(left_wrist, mouth), vector_distance(right_wrist, mouth),
        vector_distance(left_wrist, nose), vector_distance(right_wrist, nose),
        vector_distance(left_wrist, left_shoulder), vector_distance(right_wrist, right_shoulder),
        left_spread, right_spread,
        vector_distance(left_hand[:, 4, :2], left_hand[:, 8, :2]),
        vector_distance(right_hand[:, 4, :2], right_hand[:, 8, :2]),
        vector_distance(left_wrist, left_pose_wrist), vector_distance(right_wrist, right_pose_wrist),
        left_tip_to_mouth, right_tip_to_mouth,
    ], axis=1)
    left_valid = detection[:, 0] > 0.5
    right_valid = detection[:, 1] > 0.5
    pose_valid = detection[:, 2] > 0.5
    lip_valid = detection[:, 3] > 0.5
    geometry_masks = tf.stack([
        left_valid & right_valid,
        left_valid & lip_valid, right_valid & lip_valid,
        left_valid & pose_valid, right_valid & pose_valid,
        left_valid & pose_valid, right_valid & pose_valid,
        left_valid, right_valid, left_valid, right_valid,
        left_valid & pose_valid, right_valid & pose_valid,
        left_valid & lip_valid, right_valid & lip_valid,
    ], axis=1)
    geometry = tf.where(geometry_masks, tf.clip_by_value(geometry, 0.0, 10.0), 0.0)
    enhanced = tf.concat([features, velocity, acceleration, geometry], axis=1)
    return tf.ensure_shape(enhanced, [48, ENHANCED_FEATURE_COUNT])

enhanced_feature_names = (
    base_feature_names
    + [f'velocity_{name}' for name in base_feature_names[:MOTION_COORDINATE_COUNT]]
    + [f'acceleration_{name}' for name in base_feature_names[:MOTION_COORDINATE_COUNT]]
    + GEOMETRY_FEATURE_NAMES
)
sample = tf.convert_to_tensor(X_validation[0])
enhanced_sample = build_enhanced_features(sample)
np.testing.assert_allclose(
    mirror_base_features(mirror_base_features(sample)).numpy(), sample.numpy(), atol=1e-6
)
assert enhanced_sample.shape == (48, 492)
assert bool(tf.reduce_all(tf.math.is_finite(enhanced_sample)))
assert np.allclose(enhanced_sample.numpy()[0, 189:477], 0.0)
assert len(enhanced_feature_names) == 492
print('Feature and mirror contract: OK')

## 3. Freeze evaluation datasets and load both checkpoints

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def add_enhanced_features(features, label):
    return build_enhanced_features(features), label

def evaluation_dataset(features, labels, enhanced=False, mirrored=False):
    dataset = tf.data.Dataset.from_tensor_slices((features, labels))
    if mirrored:
        dataset = dataset.map(
            lambda item, label: (mirror_base_features(tf.cast(item, tf.float32)), label),
            num_parallel_calls=AUTOTUNE, deterministic=True,
        )
    if enhanced:
        dataset = dataset.map(add_enhanced_features, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

source_test = evaluation_dataset(X_test, y_test)
source_test_mirrored = evaluation_dataset(X_test, y_test, mirrored=True)
candidate_test = evaluation_dataset(X_test, y_test, enhanced=True)
candidate_test_mirrored = evaluation_dataset(X_test, y_test, enhanced=True, mirrored=True)

source_model = tf.keras.models.load_model(SOURCE_MODEL_PATH, compile=False)
candidate_model = tf.keras.models.load_model(CANDIDATE_MODEL_PATH, compile=False)
assert tuple(source_model.input_shape[1:]) == (48, 189)
assert tuple(candidate_model.input_shape[1:]) == (48, 492)
print('Frozen model input contracts: OK')

## 4. Execute the one-time comparative audit

The promotion rules were printed before this cell. No training, threshold tuning, or model
selection occurs after the holdout predictions are produced.

In [ ]:
def predict_dataset(model, dataset):
    return model.predict(dataset.map(lambda features, labels: features), verbose=0)

def probability_metrics(true_labels, probabilities):
    predictions = probabilities.argmax(axis=1)
    return {
        'accuracy': float(accuracy_score(true_labels, predictions)),
        'macro_f1': float(f1_score(true_labels, predictions, average='macro')),
        'top5_accuracy': float(top_k_accuracy_score(
            true_labels, probabilities, k=5, labels=np.arange(50)
        )),
    }

source_probabilities = predict_dataset(source_model, source_test)
source_mirrored_probabilities = predict_dataset(source_model, source_test_mirrored)
candidate_probabilities = predict_dataset(candidate_model, candidate_test)
candidate_mirrored_probabilities = predict_dataset(candidate_model, candidate_test_mirrored)

source_metrics = probability_metrics(y_test, source_probabilities)
source_mirrored_metrics = probability_metrics(y_test, source_mirrored_probabilities)
candidate_metrics = probability_metrics(y_test, candidate_probabilities)
candidate_mirrored_metrics = probability_metrics(y_test, candidate_mirrored_probabilities)
comparison = pd.DataFrame([
    {'model': 'production_189', 'view': 'original', **source_metrics},
    {'model': 'production_189', 'view': 'mirrored', **source_mirrored_metrics},
    {'model': 'candidate_492', 'view': 'original', **candidate_metrics},
    {'model': 'candidate_492', 'view': 'mirrored', **candidate_mirrored_metrics},
])
comparison.to_csv(OUTPUT_DIR / 'final_holdout_comparison.csv', index=False)
display(comparison.style.format({
    'accuracy': '{:.4f}', 'macro_f1': '{:.4f}', 'top5_accuracy': '{:.4f}'
}))

macro_f1_gain = candidate_metrics['macro_f1'] - source_metrics['macro_f1']
accuracy_gain = candidate_metrics['accuracy'] - source_metrics['accuracy']
mirror_f1_gain = candidate_mirrored_metrics['macro_f1'] - source_mirrored_metrics['macro_f1']
APPROVE_FOR_APP_INTEGRATION = bool(
    macro_f1_gain >= MINIMUM_TEST_MACRO_F1_GAIN
    and accuracy_gain >= -TEST_ACCURACY_TOLERANCE
    and mirror_f1_gain >= -TEST_MIRROR_F1_TOLERANCE
)
print('Holdout macro F1 gain:', f'{macro_f1_gain:+.4f}')
print('Holdout accuracy gain:', f'{accuracy_gain:+.4f}')
print('Holdout mirrored macro F1 gain:', f'{mirror_f1_gain:+.4f}')
print('APPROVE FOR APP INTEGRATION:', APPROVE_FOR_APP_INTEGRATION)

In [ ]:
index_to_sign = {int(index): sign for sign, index in label_map.items()}
source_predictions = source_probabilities.argmax(axis=1)
candidate_predictions = candidate_probabilities.argmax(axis=1)
source_class_f1 = f1_score(
    y_test, source_predictions, labels=np.arange(50), average=None, zero_division=0
)
candidate_class_f1 = f1_score(
    y_test, candidate_predictions, labels=np.arange(50), average=None, zero_division=0
)
per_class = pd.DataFrame({
    'class_index': np.arange(50),
    'sign': [index_to_sign[index] for index in range(50)],
    'production_f1': source_class_f1,
    'candidate_f1': candidate_class_f1,
    'f1_delta': candidate_class_f1 - source_class_f1,
}).sort_values('f1_delta', ascending=False)
per_class.to_csv(OUTPUT_DIR / 'final_holdout_per_class_delta.csv', index=False)
display(pd.concat([per_class.head(10), per_class.tail(10)]))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for axis, title, probabilities in [
    (axes[0], 'Production 189', source_probabilities),
    (axes[1], 'Candidate 492', candidate_probabilities),
]:
    matrix = confusion_matrix(
        y_test, probabilities.argmax(axis=1), labels=np.arange(50), normalize='true'
    )
    sns.heatmap(matrix, cmap='Blues', vmin=0, vmax=1, cbar=False, ax=axis)
    axis.set(title=title, xlabel='Predicted class', ylabel='True class')
plt.tight_layout()
CONFUSION_PATH = OUTPUT_DIR / 'final_holdout_confusion_comparison.png'
plt.savefig(CONFUSION_PATH, dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
def selective_metrics(probabilities, true_labels, threshold):
    predictions = probabilities.argmax(axis=1)
    confidences = probabilities.max(axis=1)
    accepted = confidences >= threshold
    return {
        'threshold': float(threshold),
        'coverage': float(accepted.mean()),
        'accepted_accuracy': (
            float((predictions[accepted] == true_labels[accepted]).mean())
            if accepted.any() else np.nan
        ),
        'accepted_count': int(accepted.sum()),
    }

source_threshold = float(source_manifest.get('confidence_threshold', 0.59))
candidate_threshold = float(candidate_manifest['confidence_threshold'])
selective = pd.DataFrame([
    {'model': 'production_189', **selective_metrics(source_probabilities, y_test, source_threshold)},
    {'model': 'candidate_492', **selective_metrics(candidate_probabilities, y_test, candidate_threshold)},
])
selective.to_csv(OUTPUT_DIR / 'final_holdout_selective_metrics.csv', index=False)
display(selective.style.format({
    'threshold': '{:.2f}', 'coverage': '{:.4f}', 'accepted_accuracy': '{:.4f}'
}))

## 5. Freeze and export the audit decision

In [ ]:
audit = {
    'experiment': 'SignLearn-09 final comparative internal holdout audit',
    'training_performed': False,
    'internal_holdout_note': (
        'Google test signers were used for this frozen comparative audit; this split has '
        'appeared in earlier project reports and is not claimed as a pristine external test.'
    ),
    'external_test_status': 'WLASL not loaded or used',
    'frozen_rules': {
        'minimum_test_macro_f1_gain': MINIMUM_TEST_MACRO_F1_GAIN,
        'test_accuracy_tolerance': TEST_ACCURACY_TOLERANCE,
        'test_mirror_f1_tolerance': TEST_MIRROR_F1_TOLERANCE,
    },
    'production_model_sha256': sha256(SOURCE_MODEL_PATH),
    'candidate_model_sha256': sha256(CANDIDATE_MODEL_PATH),
    'production_original': source_metrics,
    'production_mirrored': source_mirrored_metrics,
    'candidate_original': candidate_metrics,
    'candidate_mirrored': candidate_mirrored_metrics,
    'macro_f1_gain': macro_f1_gain,
    'accuracy_gain': accuracy_gain,
    'mirrored_macro_f1_gain': mirror_f1_gain,
    'approve_for_app_integration': APPROVE_FOR_APP_INTEGRATION,
    'selective_metrics': selective.to_dict(orient='records'),
}
AUDIT_PATH = OUTPUT_DIR / 'final_audit_decision.json'
AUDIT_PATH.write_text(json.dumps(audit, indent=2), encoding='utf-8')

RESULTS_DIR = OUTPUT_DIR / 'signlearn_09_final_audit'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for path in [
    OUTPUT_DIR / 'final_holdout_comparison.csv',
    OUTPUT_DIR / 'final_holdout_per_class_delta.csv',
    OUTPUT_DIR / 'final_holdout_selective_metrics.csv',
    CONFUSION_PATH,
    AUDIT_PATH,
]:
    shutil.copy2(path, RESULTS_DIR / path.name)
archive = shutil.make_archive(
    str(OUTPUT_DIR / 'signlearn_09_final_audit'), 'zip', root_dir=RESULTS_DIR
)
print('APPROVE FOR APP INTEGRATION:', APPROVE_FOR_APP_INTEGRATION)
print('Download and send:', archive)
print('WLASL used:', False)

## What to send back

Download `signlearn_09_final_audit.zip` and send it for review. Keep the SignLearn-08B
candidate bundle available; if the audit approves integration, the webcam application will
then be upgraded to generate the complete 492-feature input contract and tested locally.